# Netflix Content Analysis – What Makes Content Popular?

**Challenge 1:** What makes content popular on Netflix?
**Business question:** What characteristics are most common among Netflix titles?

**Popularity definition:** measured with viewer behavior from `watch_history`: number of views, unique viewers and average rating per title. A title is *popular* if it is in the top 25% by views. See `docs/project_plan.md` for the full scope.

**Notebook outline**
1. Load & Inspect Datasets
2. Clean & Merge Data
3. Exploratory Data Analysis
4. Visualizations
5. Business Insights
6. Conclusions

## 1. Load & Inspect Datasets

In this section we load the three source tables (`movies`, `users`, `watch_history`) and check their shape, data types, missing values and duplicates before any cleaning.

### 1.1 Imports and configuration

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_RAW = Path("../data/raw")

### 1.2 Load the datasets

In [2]:
movies = pd.read_csv(DATA_RAW / "movies.csv")
users = pd.read_csv(DATA_RAW / "users.csv")
watch_history = pd.read_csv(DATA_RAW / "watch_history.csv")

datasets = {"movies": movies, "users": users, "watch_history": watch_history}

for name, df in datasets.items():
    print(f"{name:<14} {df.shape[0]:>4} rows x {df.shape[1]} columns")

movies           24 rows x 7 columns
users            20 rows x 5 columns
watch_history   120 rows x 4 columns


### 1.3 Inspection helper
A small function so that every table is inspected in exactly the same way: preview, data types, missing values and duplicates.

In [3]:
def inspect_dataset(df, name, key_columns=None):
    """Print a standard inspection report for a DataFrame."""
    print(f"{'=' * 60}\n{name.upper()}  |  shape: {df.shape}\n{'=' * 60}")

    print("\nFirst rows:")
    display(df.head())

    print("Data types:")
    display(df.dtypes.rename("dtype").to_frame())

    nulls = df.isna().sum().rename("missing_values").to_frame()
    nulls["missing_%"] = (nulls["missing_values"] / len(df) * 100).round(2)
    print("Missing values:")
    display(nulls)

    print(f"Fully duplicated rows: {df.duplicated().sum()}")
    if key_columns:
        print(f"Duplicated {key_columns}: {df.duplicated(subset=key_columns).sum()}")

### 1.4 `movies`

In [4]:
inspect_dataset(movies, "movies", key_columns=["movie_id"])

MOVIES  |  shape: (24, 7)

First rows:


,movie_id,title,duration,release_year,genre,director,cast
0,1,The Last Signal,112,2018,Science Fiction,Mara Collins,Ethan Cole; Sofia Reyes
1,2,Neon Horizon,124,2021,Science Fiction,Daniel Kim,Lena Park; Marcus Reed
2,3,Orbit 9,98,2016,Science Fiction,Mara Collins,Noah Bennett; Ava Brooks
3,4,Silent Planet,135,2023,Science Fiction,Priya Shah,Sofia Reyes; Oliver Grant
4,5,Echoes of Tomorrow,107,2020,Science Fiction,Daniel Kim,Lena Park; Ethan Cole


Data types:


,dtype
movie_id,int64
title,str
duration,int64
release_year,int64
genre,str
director,str
cast,str


Missing values:


,missing_values,missing_%
movie_id,0,0.0
title,0,0.0
duration,0,0.0
release_year,0,0.0
genre,0,0.0
director,0,0.0
cast,0,0.0


Fully duplicated rows: 0
Duplicated ['movie_id']: 0


In [5]:
print("Duplicated titles:", movies["title"].duplicated().sum())
print("\nNumeric summary:")
display(movies[["duration", "release_year"]].describe())
print("Movies per genre:")
display(movies["genre"].value_counts().rename("titles").to_frame())

Duplicated titles: 0

Numeric summary:


,duration,release_year
count,24.000000,24.000000
mean,113.500000,2019.666667
std,12.507389,2.807856
min,95.000000,2015.000000
25%,103.750000,2017.750000
50%,112.500000,2020.000000
75%,122.500000,2022.000000
max,139.000000,2024.000000


Movies per genre:


,titles
genre,
Science Fiction,5
Thriller,4
Horror,3
Romance,3
Drama,3
Comedy,3
Adventure,3


### 1.5 `users`

In [6]:
inspect_dataset(users, "users", key_columns=["user_id"])

USERS  |  shape: (20, 5)

First rows:


,user_id,age,name,country,genre
0,101,19,Alex Morgan,Germany,Male
1,102,24,Beatriz Lima,Brazil,Female
2,103,31,Carlos Mendes,Portugal,Male
3,104,27,Diana Costa,Spain,Female
4,105,22,Ethan Wilson,United States,Male


Data types:


,dtype
user_id,int64
age,int64
name,str
country,str
genre,str


Missing values:


,missing_values,missing_%
user_id,0,0.0
age,0,0.0
name,0,0.0
country,0,0.0
genre,0,0.0


Fully duplicated rows: 0
Duplicated ['user_id']: 0


In [7]:
print("Numeric summary:")
display(users[["age"]].describe())
print("Values in column 'genre':", users["genre"].unique().tolist())
print("\nUsers per country:")
display(users["country"].value_counts().rename("users").to_frame())

Numeric summary:


,age
count,20.000000
mean,28.600000
std,7.659875
min,18.000000
25%,22.750000
50%,27.500000
75%,33.500000
max,45.000000


Values in column 'genre': ['Male', 'Female']

Users per country:


,users
country,
Germany,4
Brazil,4
Portugal,3
Spain,2
United States,2
United Kingdom,2
Poland,1
Canada,1
Italy,1


### 1.6 `watch_history`

In [8]:
inspect_dataset(watch_history, "watch_history")

WATCH_HISTORY  |  shape: (120, 4)

First rows:


,movie_id,user_id,rating,date
0,6,101,2,2025-01-05
1,14,102,5,2025-01-10
2,8,103,4,2025-01-15
3,1,104,4,2025-01-20
4,12,105,4,2025-01-25


Data types:


,dtype
movie_id,int64
user_id,int64
rating,int64
date,str


Missing values:


,missing_values,missing_%
movie_id,0,0.0
user_id,0,0.0
rating,0,0.0
date,0,0.0


Fully duplicated rows: 0


In [9]:
print("Rating distribution:")
display(watch_history["rating"].value_counts().sort_index().rename("records").to_frame())

print("Date range (text column, not yet converted):",
      watch_history["date"].min(), "to", watch_history["date"].max())

pair_counts = watch_history.groupby(["user_id", "movie_id"]).size()
print("\nUser-movie pairs that appear more than once:", (pair_counts > 1).sum())
print("Extra records caused by repeated pairs:", watch_history.duplicated(subset=["user_id", "movie_id"]).sum())
print("Max times the same user watched the same movie:", pair_counts.max())
print("Same user + movie + date (true duplicates):",
      watch_history.duplicated(subset=["user_id", "movie_id", "date"]).sum())

Rating distribution:


,records
rating,
2,4
3,30
4,44
5,42


Date range (text column, not yet converted): 2025-01-05 to 2026-01-06

User-movie pairs that appear more than once: 32
Extra records caused by repeated pairs: 40
Max times the same user watched the same movie: 4
Same user + movie + date (true duplicates): 0


### 1.7 Referential integrity between tables

In [10]:
orphan_movies = set(watch_history["movie_id"]) - set(movies["movie_id"])
orphan_users = set(watch_history["user_id"]) - set(users["user_id"])
unwatched_movies = set(movies["movie_id"]) - set(watch_history["movie_id"])
inactive_users = set(users["user_id"]) - set(watch_history["user_id"])

print("movie_ids in watch_history missing from movies:", orphan_movies or "none")
print("user_ids in watch_history missing from users:", orphan_users or "none")
print("Movies never watched:", unwatched_movies or "none")
print("Users with no watch records:", inactive_users or "none")

movie_ids in watch_history missing from movies: none
user_ids in watch_history missing from users: none
Movies never watched: none
Users with no watch records: none


### 1.8 Observations

- **Shapes:** `movies` has 24 rows, `users` 20 rows and `watch_history` 120 rows. This is a small sample, so findings should be treated as indications, not statistical proof.
- **Missing values:** none in any of the three tables, so no imputation is needed.
- **Keys:** `movie_id` and `user_id` are unique in their own tables, and every key in `watch_history` exists in `movies` and `users`. Every movie was watched at least once and every user has watch records.
- **Fully duplicated rows:** none.
- **Repeated user–movie pairs:** 32 pairs appear more than once (40 extra records, up to 4 times for the same pair). The dates always differ, so they are treated as **re-watches**, and each record counts as one view.
- **Column naming issue:** in `users`, the column `genre` actually contains gender (Male/Female). It will be renamed to `gender` during cleaning to avoid confusion with `movies.genre`.
- **Data types:** `date` in `watch_history` is stored as text and must be converted to datetime.
- **Value ranges look plausible:** ratings 2–5, ages 18–45, durations 95–139 minutes, release years 2015–2024.

## 2. Clean & Merge Data

Based on the inspection in section 1, this section:
1. Works on copies so the raw data stays untouched.
2. Handles missing values (verifying that none exist).
3. Fixes column naming and standardizes text, duration and date formats.
4. Parses the multi-value field `cast`.
5. Merges the three tables into one analysis-ready dataset and validates the result.

### 2.1 Work on copies

In [11]:
movies_clean = movies.copy()
users_clean = users.copy()
watch_clean = watch_history.copy()

### 2.2 Missing values
Section 1 found no missing values. We make that a hard check, so the notebook fails loudly if the raw data ever changes, and we also look for empty strings, which `isna()` does not catch.

In [12]:
for name, df in {"movies": movies_clean, "users": users_clean, "watch_history": watch_clean}.items():
    total_nulls = int(df.isna().sum().sum())
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    empty_strings = int((df[text_cols].apply(lambda col: col.str.strip() == "")).sum().sum())
    print(f"{name:<14} nulls: {total_nulls} | empty strings: {empty_strings}")
    assert total_nulls == 0 and empty_strings == 0, f"Unexpected missing data in {name}"

print("\nNo missing values: no imputation or row removal is needed.")

movies         nulls: 0 | empty strings: 0
users          nulls: 0 | empty strings: 0
watch_history  nulls: 0 | empty strings: 0

No missing values: no imputation or row removal is needed.


### 2.3 Column naming and text standardization
- `users.genre` holds gender, so it is renamed to `gender`.
- `users.country` is renamed to `user_country` to make clear it is the **viewer's** country (movies have no country).
- Leading/trailing spaces are stripped from all text columns.

In [13]:
users_clean = users_clean.rename(columns={"genre": "gender", "country": "user_country"})

for df in (movies_clean, users_clean):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].str.strip()

print("users columns:", users_clean.columns.tolist())
print("gender values:", users_clean["gender"].unique().tolist())

users columns: ['user_id', 'age', 'name', 'user_country', 'gender']
gender values: ['Male', 'Female']


### 2.4 Standardize duration
`duration` is already numeric (minutes). We make this explicit in the column name, confirm the type and check the range.

In [14]:
movies_clean = movies_clean.rename(columns={"duration": "duration_min"})
movies_clean["duration_min"] = movies_clean["duration_min"].astype(int)

print(movies_clean["duration_min"].describe()[["min", "max", "mean"]].round(1).to_dict())
assert movies_clean["duration_min"].between(30, 300).all(), "Duration outside a plausible range"

{'min': 95.0, 'max': 139.0, 'mean': 113.5}


### 2.5 Standardize dates
`date` is stored as text (`YYYY-MM-DD`). It is converted to a real datetime, renamed `watch_date`, and a `watch_month` column is added for trend analysis later.

In [15]:
watch_clean["watch_date"] = pd.to_datetime(watch_clean["date"], format="%Y-%m-%d", errors="raise")
watch_clean = watch_clean.drop(columns="date")
watch_clean["watch_month"] = watch_clean["watch_date"].dt.to_period("M").astype(str)

print(watch_clean["watch_date"].dtype)
print("Period:", watch_clean["watch_date"].min().date(), "to", watch_clean["watch_date"].max().date())
print("Months covered:", watch_clean["watch_month"].nunique())

datetime64[us]
Period: 2025-01-05 to 2026-01-06
Months covered: 13


### 2.6 Parse multi-value fields
`cast` stores several actors in one cell, separated by `;`. Other columns (`genre`, `director`) hold a single value per movie.

We keep the original `cast` column and also create a **long table** `movie_cast` (one row per movie–actor pair), which is the right shape for counting how often each actor appears.

In [16]:
movies_clean["cast_list"] = movies_clean["cast"].str.split(";").apply(lambda names: [n.strip() for n in names])
movies_clean["cast_size"] = movies_clean["cast_list"].str.len()

movie_cast = (
    movies_clean[["movie_id", "cast_list"]]
    .explode("cast_list")
    .rename(columns={"cast_list": "actor"})
    .reset_index(drop=True)
)

print("Cast size per movie:", movies_clean["cast_size"].value_counts().to_dict())
print("Movie–actor rows:", len(movie_cast), "| distinct actors:", movie_cast["actor"].nunique())
display(movie_cast.head())

Cast size per movie: {2: 24}
Movie–actor rows: 48 | distinct actors: 8


,movie_id,actor
0,1,Ethan Cole
1,1,Sofia Reyes
2,2,Lena Park
3,2,Marcus Reed
4,3,Noah Bennett


### 2.7 Merge the tables
`watch_history` is the central (fact) table: each row is one view. Movie and user attributes are attached with **left joins**, validated as many-to-one, so the number of rows must stay at 120.

In [17]:
movie_columns = ["movie_id", "title", "genre", "duration_min", "release_year", "director", "cast"]
user_columns = ["user_id", "age", "gender", "user_country"]

merged = (
    watch_clean
    .merge(movies_clean[movie_columns], on="movie_id", how="left", validate="many_to_one")
    .merge(users_clean[user_columns], on="user_id", how="left", validate="many_to_one")
)

column_order = [
    "user_id", "movie_id", "title", "genre", "duration_min", "release_year", "director", "cast",
    "rating", "watch_date", "watch_month", "age", "gender", "user_country",
]
merged = merged[column_order].sort_values("watch_date").reset_index(drop=True)

print("Merged shape:", merged.shape)
display(merged.head())

Merged shape: (120, 14)


,user_id,movie_id,title,genre,duration_min,release_year,director,cast,rating,watch_date,watch_month,age,gender,user_country
0,101,6,Midnight Runaway,Thriller,101,2017,Lucas Meyer,Mia Foster; Noah Bennett,2,2025-01-05,2025-01,19,Male,Germany
1,102,14,Letters in December,Romance,114,2020,Isabel Costa,Mia Foster; Ethan Cole,5,2025-01-10,2025-01,24,Female,Brazil
2,103,8,Cold Evidence,Thriller,109,2019,Lucas Meyer,Marcus Reed; Mia Foster,4,2025-01-15,2025-01,31,Male,Portugal
3,104,1,The Last Signal,Science Fiction,112,2018,Mara Collins,Ethan Cole; Sofia Reyes,4,2025-01-20,2025-01,27,Female,Spain
4,105,12,Red Harvest,Horror,116,2018,Priya Shah,Marcus Reed; Lena Park,4,2025-01-25,2025-01,22,Male,United States


### 2.8 Validate the merged dataset

In [18]:
assert len(merged) == len(watch_history), "Row count changed during merge"
assert merged.isna().sum().sum() == 0, "Nulls introduced by merge"
assert merged["rating"].between(1, 5).all(), "Rating outside 1-5"
assert set(merged["movie_id"]) == set(movies["movie_id"]), "Some movies were lost"
assert set(merged["user_id"]) == set(users["user_id"]), "Some users were lost"
assert (merged.groupby("movie_id").size() == watch_history.groupby("movie_id").size()).all(), "Views per movie changed"

print("All validation checks passed.")
print("\nFinal data types:")
display(merged.dtypes.rename("dtype").to_frame())

All validation checks passed.

Final data types:


,dtype
user_id,int64
movie_id,int64
title,str
genre,str
duration_min,int64
release_year,int64
director,str
cast,str
rating,int64
watch_date,datetime64[us]


### 2.9 Save processed data

In [19]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

merged.to_csv(PROCESSED / "merged_watch_data.csv", index=False)
movie_cast.to_csv(PROCESSED / "movie_cast.csv", index=False)

print("Saved:", sorted(p.name for p in PROCESSED.glob("*.csv")))

Saved: ['merged_watch_data.csv', 'movie_cast.csv']


### 2.10 Summary of cleaning decisions

| Issue | Decision |
|---|---|
| Missing values | None found (verified with a check); no imputation or removal |
| `users.genre` contains gender | Renamed to `gender` |
| `users.country` is the viewer's country | Renamed to `user_country` |
| `duration` | Already numeric; renamed `duration_min` (minutes) and range-checked |
| `date` stored as text | Converted to datetime (`watch_date`); added `watch_month` |
| Multi-value `cast` | Kept original column and created long table `movie_cast` (one row per movie–actor) |
| Repeated user–movie pairs | Kept: they have different dates, so each record is one view (re-watch) |
| Merge | `watch_history` as base table, left joins validated many-to-one; 120 rows preserved |

**Output:** `data/processed/merged_watch_data.csv` (120 rows × 14 columns) and `data/processed/movie_cast.csv`.

**Next step:** Exploratory Data Analysis on the merged dataset.